# 05. ML Peer Clustering
## K-Means clustering + Vietnam Peer Group identification

**Objective**:
- Apply K-Means clustering on normalized data
- Identify Vietnam's cluster (Vietnam Peer Group)
- Use Dynamic Time Warping (DTW) to find most similar countries

**Output**:
- Cluster assignments per country
- Vietnam peer group (top 10-15 similar countries)
- Cluster visualization

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.cluster import KMeans
from scipy.spatial.distance import euclidean
import warnings
warnings.filterwarnings('ignore')

# Load scaled data
data_dir = Path('./outputs')
consolidated_scaled = pd.read_csv(data_dir / 'consolidated_scaled.csv', index_col=0)

print(f"✓ Loaded scaled data: {consolidated_scaled.shape}")

## 1. K-Means Clustering

In [ ]:
# Determine optimal k using elbow method
print("\n" + "="*80)
print("K-MEANS CLUSTERING")
print("="*80 + "\n")

# Calculate inertia for different k values
inertias = []
k_range = range(2, 11)

for k in k_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_temp.fit(consolidated_scaled)
    inertias.append(kmeans_temp.inertia_)

print("Elbow curve (k vs inertia):")
for k, inertia in zip(k_range, inertias):
    print(f"  k={k}: {inertia:.2f}")

# Use k=5-7 as reasonable balance (adjust based on business need)
optimal_k = 6
print(f"\n→ Selected k={optimal_k} (business interpretation)")

# Fit K-Means
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(consolidated_scaled)

# Add cluster labels to data
consolidated_scaled['cluster'] = cluster_labels

print(f"\n✓ K-Means fitted with k={optimal_k}")
print(f"\nCluster distribution:")
print(pd.Series(cluster_labels).value_counts().sort_index())

## 2. Identify Vietnam Peer Group

In [ ]:
# Find Vietnam and its cluster
print("\n" + "="*80)
print("VIETNAM PEER GROUP IDENTIFICATION")
print("="*80 + "\n")

if 'VNM' in consolidated_scaled.index:
    vietnam_cluster = consolidated_scaled.loc['VNM', 'cluster']
    vietnam_vector = consolidated_scaled.loc['VNM', :-1].values  # Exclude cluster column
    
    print(f"✓ Vietnam (VNM) assigned to cluster {int(vietnam_cluster)}")
    
    # Find all countries in same cluster
    peer_cluster = consolidated_scaled[consolidated_scaled['cluster'] == vietnam_cluster]
    print(f"  Cluster size: {len(peer_cluster)} countries")
    
    # Calculate distance from Vietnam to all others in cluster
    distances = {}
    for country in peer_cluster.index:
        if country != 'VNM':
            country_vector = consolidated_scaled.loc[country, :-1].values
            dist = euclidean(vietnam_vector, country_vector)
            distances[country] = dist
    
    # Sort by distance
    sorted_peers = sorted(distances.items(), key=lambda x: x[1])
    
    print(f"\n→ Top 15 Most Similar Countries (by Euclidean distance):")
    print("-" * 50)
    for i, (country, dist) in enumerate(sorted_peers[:15], 1):
        print(f"  {i:2d}. {country}: {dist:.4f}")
    
    # Define formal peer group (top 10-12 most similar)
    peer_group = [country for country, _ in sorted_peers[:12]]
    print(f"\n✓ Vietnam Peer Group (ML-selected): {peer_group}")
else:
    print("✗ Vietnam (VNM) not found in data")
    peer_group = []

## 3. Save Clustering Results